In [1]:
# Install (Colab only)
!pip install torch pandas numpy scikit-learn

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [2]:
SEQ_FEATURES = [
    "ball_runs","ball_wicket","is_boundary",
    "runs_so_far","wickets_fallen","balls_bowled",
    "required_runs","balls_remaining",
    "current_run_rate","required_run_rate","run_rate_diff",
    "resources_remaining",
    "runs_last_6","wickets_last_12","rr_acceleration",
    "pressure_index_b","match_phase_enc"
]

TARGET = "target_win"
TEST_SEASONS = ["2024","2025"]
VAL_SEASON = "2023"

In [18]:
df = pd.read_csv("ipl_phase3.csv")
df["season"] = df["season"].astype(str)

df = df.sort_values(["matchId","over","ball"]).reset_index(drop=True)

# Fix NaN / Inf issues
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# Clip unstable features
clip_cols = [
    "current_run_rate","required_run_rate",
    "run_rate_diff","rr_acceleration","pressure_index_b"
]

for col in clip_cols:
    df[col] = df[col].clip(-50, 50)

df.head()

/tmp/ipykernel_1779/3004740291.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ipl_phase3.csv")


,matchId,season,over,ball,runs_so_far,wickets_fallen,wickets_in_hand,balls_bowled,balls_remaining,overs_completed,...,is_boundary,runs_last_6,runs_last_12,wickets_last_12,boundaries_last_6,rr_acceleration,pressure_index_a,pressure_index_b,phase_x_rrr,phase_x_wkts
0,335982,2007/08,0,1,1,0,10,1,119,0.166667,...,0,1.0,1.0,0.0,0.0,-5.0,1.865546,1.865546,0.0,0
1,335982,2007/08,0,2,2,0,10,1,119,0.166667,...,0,2.0,2.0,0.0,0.0,-10.0,0.928571,1.857143,0.0,0
2,335982,2007/08,0,3,2,0,10,2,118,0.333333,...,0,2.0,2.0,0.0,0.0,-4.0,1.872881,1.872881,0.0,0
3,335982,2007/08,0,4,3,0,10,3,117,0.500000,...,0,3.0,3.0,0.0,0.0,-3.0,1.880342,1.880342,0.0,0
4,335982,2007/08,0,5,4,0,10,4,116,0.666667,...,0,4.0,4.0,0.0,0.0,-2.0,1.887931,1.887931,0.0,0


In [19]:
from sklearn.preprocessing import StandardScaler

train_mask = ~df["season"].isin(TEST_SEASONS + [VAL_SEASON])

scaler = StandardScaler()

# Fit only on training data
scaler.fit(df.loc[train_mask, SEQ_FEATURES])

# Transform full dataset
df.loc[:, SEQ_FEATURES] = scaler.transform(df[SEQ_FEATURES])

/tmp/ipykernel_1779/493802183.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.44058897 -0.44058897 -0.44058897 ... -0.44058897 -0.44058897
  2.26968915]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, SEQ_FEATURES] = scaler.transform(df[SEQ_FEATURES])
/tmp/ipykernel_1779/493802183.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.54752665 -1.52604193 -1.52604193 ...  2.74941734  2.77090206
  2.89981038]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, SEQ_FEATURES] = scaler.transform(df[SEQ_FEATURES])
/tmp/ipykernel_1779/493802183.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14593466 -1.14593466 -1.14593466 ...  0.71630452  0.71630452
  0.716304

In [41]:
sequences, labels, lengths, match_ids, seasons = [], [], [], [], []

for mid, g in df.groupby("matchId"):
    g = g.sort_values(["over","ball"])

    if len(g) < 30:
        continue

    cut = np.random.randint(30, len(g))  # random cut for live prediction
    g_cut = g.iloc[:cut]

    seq = g_cut[SEQ_FEATURES].values.astype(np.float32)
    lbl = int(g[TARGET].iloc[-1])

    sequences.append(seq)
    labels.append(lbl)
    lengths.append(len(seq))
    match_ids.append(mid)
    seasons.append(g["season"].iloc[0])

# convert to numpy for indexing
labels = np.array(labels)
lengths = np.array(lengths)

In [42]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class MatchDataset(Dataset):
    def __init__(self, seqs, labels, lengths):
        self.seqs = seqs
        self.labels = labels
        self.lengths = lengths

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            torch.tensor(self.seqs[i]),
            torch.tensor(self.labels[i], dtype=torch.float32),
            torch.tensor(self.lengths[i])
        )


def collate(batch):
    seqs, labels, lengths = zip(*batch)

    order = np.argsort(lengths)[::-1]

    seqs = [seqs[i] for i in order]
    labels = torch.stack([labels[i] for i in order])
    lengths = torch.tensor([lengths[i] for i in order])

    padded = pad_sequence(seqs, batch_first=True)
    return padded, labels, lengths

In [43]:
train_idx, val_idx, test_idx = [], [], []

for i, s in enumerate(seasons):
    if s in TEST_SEASONS:
        test_idx.append(i)
    elif s == VAL_SEASON:
        val_idx.append(i)
    else:
        train_idx.append(i)

def subset(idxs):
    return [sequences[i] for i in idxs], labels[idxs], lengths[idxs]

train_ds = MatchDataset(*subset(train_idx))
val_ds   = MatchDataset(*subset(val_idx))
test_ds  = MatchDataset(*subset(test_idx))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate)

In [44]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(SEQ_FEATURES), 64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True)
        _, (h, _) = self.lstm(packed)
        return self.fc(h[-1]).squeeze()

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(20):
    model.train()
    total_loss = 0

    for x, y, l in train_loader:
        x, y, l = x.to(device), y.to(device), l.to(device)

        optimizer.zero_grad()
        out = model(x, l)
        loss = criterion(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 19.8246
Epoch 2, Loss: 19.4953
Epoch 3, Loss: 19.1496
Epoch 4, Loss: 18.7489
Epoch 5, Loss: 18.3122
Epoch 6, Loss: 17.7571
Epoch 7, Loss: 17.1212
Epoch 8, Loss: 16.4401
Epoch 9, Loss: 15.9258
Epoch 10, Loss: 15.5287
Epoch 11, Loss: 15.3045
Epoch 12, Loss: 15.1972
Epoch 13, Loss: 15.1367
Epoch 14, Loss: 15.0751
Epoch 15, Loss: 15.0235
Epoch 16, Loss: 14.9576
Epoch 17, Loss: 14.8740
Epoch 18, Loss: 14.9206
Epoch 19, Loss: 14.8234
Epoch 20, Loss: 14.8434


In [46]:
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

model.eval()
probs, true = [], []

with torch.no_grad():
    for x, y, l in test_loader:
        x, l = x.to(device), l.to(device)
        out = model(x, l)

        out = torch.sigmoid(out)

        probs.extend(out.cpu().numpy())
        true.extend(y.numpy())

probs = np.array(probs)
true = np.array(true)
preds = (probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(true, preds))
print("Log Loss:", log_loss(true, probs))
print("AUC:", roc_auc_score(true, probs))

Accuracy: 0.8285714285714286
Log Loss: 0.449826019244457
AUC: 0.9101307189542484


In [47]:
import pickle

torch.save(model.state_dict(), "lstm_model.pt")

with open("lstm_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("lstm_features.pkl", "wb") as f:
    pickle.dump(SEQ_FEATURES, f)